# 🔍 Paso 0 — Inspección de Datos Locales (Sturm)

**Corre este notebook ANTES que el manifest builder.**  
Propósito: entender exactamente qué tienes, cómo están nombrados los archivos, y qué columnas existen. El output de este notebook determina los ajustes necesarios en el manifest builder.

---
### Instrucciones
1. Ajusta las rutas en la celda **CONFIG** (sección 1)
2. Corre todas las celdas en orden
3. Copia los outputs en un mensaje para ajustar el manifest builder

## 1. CONFIG — Ajusta estas rutas a tu sistema local

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  AJUSTA ESTAS RUTAS
# ─────────────────────────────────────────────────────────────────────────────
# Carpeta donde tienes las imágenes brightfield
IMAGES_PATH = Path(r"/Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images")  
# Archivo CSV del Shiny app (puede ser uno o varios — pon la carpeta)
CSV_PATH    = Path(r'/Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv') 


print('Rutas configuradas:')
print(f'  Imágenes: {IMAGES_PATH}')
print(f'  CSV:      {CSV_PATH}')
print(f'  ¿Imágenes existe?: {IMAGES_PATH.exists()}')
print(f'  ¿CSV existe?:      {CSV_PATH.exists()}')

Rutas configuradas:
  Imágenes: /Volumes/SanDisk SSD 1TB/Storage/Data/Cellular_Lifespan_Study_Brightfield_Images
  CSV:      /Users/JCB/Documentos/Proyecto Integrador/data/Lifespan_Study_Data.csv
  ¿Imágenes existe?: True
  ¿CSV existe?:      True


## 2. Explorar estructura de la carpeta de imágenes

In [3]:
# ── Estructura de directorios ──────────────────────────────────────────────────
print('=' * 60)
print('ESTRUCTURA DE CARPETA DE IMAGENES')
print('=' * 60)

if IMAGES_PATH.is_dir():
    try:
        # Ver subcarpetas (si las hay)
        subdirs = [d for d in IMAGES_PATH.iterdir() if d.is_dir()]
        if subdirs:
            print(f'\nSubcarpetas encontradas: {len(subdirs)}')
            for d in sorted(subdirs)[:20]:
                try:
                    n_files = len(list(d.glob('*')))
                except PermissionError:
                    n_files = -1
                if n_files >= 0:
                    print(f'   {d.name}/  ({n_files} archivos)')
                else:
                    print(f'   {d.name}/  (sin permiso para contar archivos)')
            if len(subdirs) > 20:
                print(f'   ... y {len(subdirs)-20} mas')
        else:
            print('\nNo hay subcarpetas - carpeta plana')

        # Contar archivos por extension
        all_files = list(IMAGES_PATH.rglob('*.*'))
        from collections import Counter
        ext_counts = Counter(f.suffix.lower() for f in all_files)
        print(f'\nArchivos por extension:')
        for ext, count in sorted(ext_counts.items(), key=lambda x: -x[1]):
            print(f'   {ext or "sin extension":10s}: {count:5d}')

        print(f'\nTotal archivos: {len(all_files)}')

    except PermissionError as e:
        print('\nERROR DE PERMISOS: macOS bloquea acceso a esa carpeta.')
        print(f'Detalle: {e}')
        print('\nSolucion en macOS:')
        print('1) System Settings > Privacy & Security > Files and Folders')
        print('2) Habilita acceso para VS Code al volumen externo (External Volumes).')
        print('3) Si sigue igual, habilita Full Disk Access para VS Code.')
        print('4) Cierra y abre VS Code, luego reintenta esta celda.')

elif IMAGES_PATH.is_file():
    print(f'\nEs un archivo (probablemente .zip): {IMAGES_PATH.name}')
    print(f'   Tamano: {IMAGES_PATH.stat().st_size / 1e9:.2f} GB')
else:
    print('\nRuta no encontrada - verifica IMAGES_PATH')

ESTRUCTURA DE CARPETA DE IMAGENES

Subcarpetas encontradas: 4
   PhaseI/  (28 archivos)
   PhaseII/  (79 archivos)
   PhaseIII/  (12 archivos)
   PhaseV/  (40 archivos)

Archivos por extension:
   .jpg      :  2182
   .jpeg     :   633
   .tiff     :   452
   .pdf      :   226
   sin extension:    51
   .xlsx     :    34
   .11-31    :     1
   .11-11    :     1

Total archivos: 3580


In [4]:
# ── Nombres de archivo de imágenes (los primeros 30) ─────────────────────────
print('=' * 60)
print('NOMBRES DE ARCHIVO — PRIMEROS 30 (crítico para el join)')
print('=' * 60)

image_extensions = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}

if IMAGES_PATH.is_dir():
    img_files = [f for f in IMAGES_PATH.rglob('*') if f.suffix.lower() in image_extensions]
    img_files_sorted = sorted(img_files)[:30]

    for f in img_files_sorted:
        # Mostrar ruta relativa para ver estructura
        try:
            rel = f.relative_to(IMAGES_PATH)
        except:
            rel = f.name
        print(f'  {rel}')

    print(f'\n  Total imágenes encontradas: {len(img_files)}')

    # Analizar estructura del nombre (split por _ y /)
    print('\n🔍 Análisis de naming convention (primeros 10 stems):')
    for f in img_files_sorted[:10]:
        stem = f.stem
        parts = stem.split('_')
        print(f'  Nombre: {stem}')
        print(f'  Partes: {parts}')
        print()

NOMBRES DE ARCHIVO — PRIMEROS 30 (crítico para el join)
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, Ctrl.jpg
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, DEX.jpg
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, MitoQ + DEX.jpg
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, MitoQ.jpg
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, NAC + DEX.jpg
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, NAC.jpg
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, Puls.jpg
  PhaseI/P10 2017-11-10.11-11/hFB1-m P10  t175 4 days growth 10x, a-ketoglutarate.jpg
  PhaseI/P10 2017-11-10.11-11/hFB2-f P10  t175 4 days growth 10x, Ctrl.jpg
  PhaseI/P10 2017-11-10.11-11/hFB2-f P10  t175 4 days growth 10x, DEX.jpg
  PhaseI/P10 2017-11-10.11-11/hFB2-f P10  t175 4 days growth 10x, MitoQ + DEX.jpg
  PhaseI/P10 2017-11-10.11-11/hFB2-f P10  t175 4 days growth 10x, MitoQ.jpg
  PhaseI/P10 20

## 3. Explorar el CSV del Shiny app

In [5]:
# ── Listar CSVs disponibles ───────────────────────────────────────────────────
print('=' * 60)
print('ARCHIVOS CSV DISPONIBLES')
print('=' * 60)

csv_files = []

if CSV_PATH.is_file():
    csv_files = [CSV_PATH]
elif CSV_PATH.is_dir():
    csv_files = list(CSV_PATH.rglob('*.csv')) + \
                list(CSV_PATH.rglob('*.xlsx')) + \
                list(CSV_PATH.rglob('*.tsv'))

print(f'\n{len(csv_files)} archivos encontrados:')
for f in sorted(csv_files):
    size_kb = f.stat().st_size / 1e3
    print(f'  [{size_kb:8.1f} KB]  {f.name}')

ARCHIVOS CSV DISPONIBLES

1 archivos encontrados:
  [  1690.4 KB]  Lifespan_Study_Data.csv


In [6]:
# ── Inspeccionar cada CSV ─────────────────────────────────────────────────────
print('=' * 60)
print('CONTENIDO DE CADA CSV (columnas + primeras 3 filas)')
print('=' * 60)

loaded_dfs = {}

for f in sorted(csv_files):
    print(f'\n\n── {f.name} ──────────────────────────────────')
    try:
        if f.suffix == '.xlsx':
            df = pd.read_excel(f)
        elif f.suffix == '.tsv':
            df = pd.read_csv(f, sep='\t')
        else:
            # Intentar diferentes separadores
            for sep in [',', ';', '\t']:
                try:
                    df = pd.read_csv(f, sep=sep, nrows=5)
                    if df.shape[1] > 1:
                        break
                except:
                    continue

        print(f'  Shape: {df.shape}')
        print(f'  Columnas ({len(df.columns)}):')
        for i, col in enumerate(df.columns):
            dtype = str(df[col].dtype)
            n_unique = df[col].nunique()
            n_null = df[col].isna().sum()
            sample_vals = df[col].dropna().unique()[:3]
            print(f'    [{i:3d}] {col:<35s} dtype={dtype:<10s} unique={n_unique:5d} null={n_null:4d}  ejemplo: {sample_vals}')

        loaded_dfs[f.name] = df

    except Exception as e:
        print(f'  ❌ Error: {e}')

CONTENIDO DE CADA CSV (columnas + primeras 3 filas)


── Lifespan_Study_Data.csv ──────────────────────────────────
  Shape: (5, 271)
  Columnas (271):
    [  0] Sample                              dtype=int64      unique=    5 null=   0  ejemplo: [1 2 3]
    [  1] Cell_Line_Group                     dtype=str        unique=    1 null=   0  ejemplo: <StringArray>
['hFB1_1_Normal_Control_21']
Length: 1, dtype: str
    [  2] Unique_Variable_Name                dtype=str        unique=    5 null=   0  ejemplo: <StringArray>
['hFB1_1_Normal_Control_21_P3T', 'hFB1_1_Normal_Control_21_P4T',
 'hFB1_1_Normal_Control_21_P5T']
Length: 3, dtype: str
    [  3] Cell_Line                           dtype=str        unique=    1 null=   0  ejemplo: <StringArray>
['hFB1']
Length: 1, dtype: str
    [  4] Cell_line_new                       dtype=str        unique=    1 null=   0  ejemplo: <StringArray>
['HC5']
Length: 1, dtype: str
    [  5] Cell_line_group_new                 dtype=str        unique=  

In [7]:
# ── Buscar columnas clave en todos los CSVs ───────────────────────────────────
print('=' * 60)
print('BÚSQUEDA DE COLUMNAS CLAVE (sample_id, PDL, donor, passage)')
print('=' * 60)

KEYWORDS = ['sample', 'id', 'pdl', 'donor', 'passage', 'treatment',
            'phase', 'condition', 'telomere', 'mtdna', 'batch', 'cell_line',
            'timepoint', 'time', 'replicate', 'plate']

for fname, df in loaded_dfs.items():
    matches = [col for col in df.columns
               if any(kw.lower() in col.lower() for kw in KEYWORDS)]
    if matches:
        print(f'\n📋 {fname}:')
        for col in matches:
            sample_vals = df[col].dropna().unique()[:5]
            print(f'   {col:<35s} → {sample_vals}')

BÚSQUEDA DE COLUMNAS CLAVE (sample_id, PDL, donor, passage)

📋 Lifespan_Study_Data.csv:
   Sample                              → [1 2 3 4 5]
   Cell_Line_Group                     → <StringArray>
['hFB1_1_Normal_Control_21']
Length: 1, dtype: str
   Cell_Line                           → <StringArray>
['hFB1']
Length: 1, dtype: str
   Cell_line_new                       → <StringArray>
['HC5']
Length: 1, dtype: str
   Cell_line_group_new                 → <StringArray>
['HC5_sp1_Normal_Control_ox21']
Length: 1, dtype: str
   Clinical_Condition                  → <StringArray>
['Normal']
Length: 1, dtype: str
   Donor_Age                           → [29]
   Donor_Age_with_gestation            → [30]
   Replicate_Line                      → [1]
   ShinyApp_Replicate_Line             → [0]
   Treatment                           → <StringArray>
['Control_21']
Length: 1, dtype: str
   Treatments                          → <StringArray>
['Control']
Length: 1, dtype: str
   Treatment_descripti

In [8]:
# ── Si hay columna PDL: estadísticas básicas ──────────────────────────────────
print('=' * 60)
print('ANÁLISIS DE PDL (si existe)')
print('=' * 60)

for fname, df in loaded_dfs.items():
    pdl_cols = [c for c in df.columns if 'pdl' in c.lower() or 'population' in c.lower()]
    if pdl_cols:
        print(f'\n📋 {fname}:')
        for col in pdl_cols:
            try:
                vals = pd.to_numeric(df[col], errors='coerce')
                print(f'  {col}:')
                print(f'    N válidos: {vals.notna().sum()}')
                print(f'    Min: {vals.min():.2f}  Max: {vals.max():.2f}  Mean: {vals.mean():.2f}')
                # PDL por donante si hay columna de donante
                donor_cols = [c for c in df.columns if 'donor' in c.lower() or 'cell_line' in c.lower()]
                if donor_cols:
                    donor_col = donor_cols[0]
                    print(f'\n  PDL por {donor_col}:')
                    print(df.groupby(donor_col)[col].agg(['min','max','count']).to_string())
            except Exception as e:
                print(f'    Error: {e}')

ANÁLISIS DE PDL (si existe)

📋 Lifespan_Study_Data.csv:
  Population_Doublings_DI:
    N válidos: 5
    Min: 0.00  Max: 10.82  Mean: 5.13

  PDL por Cell_Line_Group:
                          min    max  count
Cell_Line_Group                            
hFB1_1_Normal_Control_21  0.0  10.82      5
  Population_Doubling_Time_DI:
    N válidos: 5
    Min: 0.00  Max: 54.37  Mean: 29.94

  PDL por Cell_Line_Group:
                          min    max  count
Cell_Line_Group                            
hFB1_1_Normal_Control_21  0.0  54.37      5
  MiAge_Population_Doublings:
    N válidos: 5
    Min: 37.22  Max: 47.90  Mean: 42.29

  PDL por Cell_Line_Group:
                            min   max  count
Cell_Line_Group                             
hFB1_1_Normal_Control_21  37.22  47.9      5


In [9]:
# ── Resumen final para copiar ─────────────────────────────────────────────────
print('=' * 60)
print('📋 RESUMEN PARA COPIAR AL MANIFEST BUILDER')
print('   Copia este bloque y compártelo para ajustar el pipeline')
print('=' * 60)

print('\n### IMÁGENES ###')
if IMAGES_PATH.is_dir():
    img_files = [f for f in IMAGES_PATH.rglob('*') if f.suffix.lower() in {'.tif','.tiff','.png','.jpg'}]
    print(f'Total imágenes: {len(img_files)}')
    if img_files:
        print(f'Ejemplos de nombres:')
        for f in sorted(img_files)[:5]:
            print(f'  {f.name}')
        print(f'¿Subcarpetas?: {len([d for d in IMAGES_PATH.iterdir() if d.is_dir()]) > 0}')

print('\n### CSV ###')
for fname, df in loaded_dfs.items():
    print(f'Archivo: {fname}')
    print(f'Shape: {df.shape}')
    print(f'Columnas: {list(df.columns)}')
    print()

📋 RESUMEN PARA COPIAR AL MANIFEST BUILDER
   Copia este bloque y compártelo para ajustar el pipeline

### IMÁGENES ###
Total imágenes: 2634
Ejemplos de nombres:
  hFB1-m P10  t175 4 days growth 10x, Ctrl.jpg
  hFB1-m P10  t175 4 days growth 10x, DEX.jpg
  hFB1-m P10  t175 4 days growth 10x, MitoQ + DEX.jpg
  hFB1-m P10  t175 4 days growth 10x, MitoQ.jpg
  hFB1-m P10  t175 4 days growth 10x, NAC + DEX.jpg
¿Subcarpetas?: True

### CSV ###
Archivo: Lifespan_Study_Data.csv
Shape: (5, 271)
Columnas: ['Sample', 'Cell_Line_Group', 'Unique_Variable_Name', 'Cell_Line', 'Cell_line_new', 'Cell_line_group_new', 'Unique_variable_name_new', 'Cell_Type', 'Sex', 'Clinical_Condition', 'Donor_Age', 'Donor_Age_with_gestation', 'Study_Part', 'Replicate_Line', 'ShinyApp_Replicate_Line', 'Treatment', 'Treatments', 'Treatment_description', 'Percent_Oxygen', 'Date.1', 'Date_of_Passage', 'Time_of_Passage', 'Date_Time_of_Passage', 'Notes', 'Passage', 'Days_Grown', 'Days_Grown_with_prestudy_days_grown', 'Days_af

## 4. (Opcional) Descomprimir .zip si aplica

In [ ]:
# Corre esta celda solo si tus imágenes están en un .zip
import zipfile

ZIP_PATH    = Path(r'RUTA/AL/ARCHIVO.zip')   # ← ajusta si aplica
EXTRACT_TO  = Path(r'RUTA/DESTINO')           # ← ajusta si aplica

if ZIP_PATH.exists() and ZIP_PATH.suffix == '.zip':
    print(f'Descomprimiendo {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/1e9:.2f} GB)...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        # Primero ver qué hay adentro
        names = zf.namelist()
        print(f'  {len(names)} archivos en el zip')
        print(f'  Primeros 10:')
        for n in names[:10]:
            print(f'    {n}')

        # Descomentar para extraer:
        # EXTRACT_TO.mkdir(parents=True, exist_ok=True)
        # zf.extractall(EXTRACT_TO)
        # print(f'✅ Extraído en: {EXTRACT_TO}')
else:
    print('No hay .zip que descomprimir (o ruta no configurada)')